In [2]:
# ========== 配置区 ==========
EMAIL = "rerezhang22@gmail.com"   # 改成你的邮箱，NCBI要求标注身份，随便什么邮箱都行

# 挂载Google Drive（运行后会弹授权，点同意）
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/wound-agent/data"
os.makedirs(SAVE_DIR, exist_ok=True)
print("保存目录:", SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
保存目录: /content/drive/MyDrive/wound-agent/data


In [3]:
import requests, time, json
import xml.etree.ElementTree as ET
import pandas as pd

BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# 检索式：慢性难愈创面 × 智能敷料/水凝胶 × 促愈合机制，2019年至今
QUERY = (
    '("chronic wound"[Title/Abstract] OR "diabetic foot ulcer"[Title/Abstract] '
    'OR "non-healing wound"[Title/Abstract] OR "nonhealing wound"[Title/Abstract]) '
    'AND (hydrogel[Title/Abstract] OR "smart dressing"[Title/Abstract] '
    'OR scaffold[Title/Abstract] OR nanofiber[Title/Abstract]) '
    'AND (angiogenesis[Title/Abstract] OR antibacterial[Title/Abstract] '
    'OR "wound healing"[Title/Abstract]) '
    'AND ("2019/01/01"[Date - Publication] : "2026/12/31"[Date - Publication])'
)

r = requests.get(f"{BASE}/esearch.fcgi", params={
    "db": "pubmed", "term": QUERY, "retmax": 50,
    "retmode": "json", "sort": "relevance", "email": EMAIL,
}, timeout=30)
result = r.json()["esearchresult"]
pmids = result["idlist"]
print(f"命中总数: {result['count']} 篇，本批抓取: {len(pmids)} 篇")

命中总数: 784 篇，本批抓取: 50 篇


In [4]:
def fetch_batch(id_list):
    r = requests.get(f"{BASE}/efetch.fcgi", params={
        "db": "pubmed", "id": ",".join(id_list), "retmode": "xml", "email": EMAIL,
    }, timeout=60)
    root = ET.fromstring(r.content)
    records = []
    for art in root.findall(".//PubmedArticle"):
        title_node = art.find(".//ArticleTitle")
        records.append({
            "pmid": art.findtext(".//PMID"),
            "title": "".join(title_node.itertext()) if title_node is not None else "",
            "journal": art.findtext(".//Journal/Title", ""),
            "year": art.findtext(".//JournalIssue/PubDate/Year") or "",
            "doi": next((a.text for a in art.findall(".//ArticleId") if a.get("IdType")=="doi"), ""),
            "pmc": next((a.text for a in art.findall(".//ArticleId") if a.get("IdType")=="pmc"), ""),
            "abstract": " ".join("".join(a.itertext()) for a in art.findall(".//Abstract/AbstractText")),
        })
    return records

# 分批抓取（每批20篇，防止超时）
all_records = []
for i in range(0, len(pmids), 20):
    all_records += fetch_batch(pmids[i:i+20])
    time.sleep(0.4)  # 礼貌限速，NCBI要求每秒不超过3次请求
    print(f"已抓取 {len(all_records)} / {len(pmids)}")

df = pd.DataFrame(all_records)
df["has_fulltext"] = df["pmc"] != ""
df.to_csv(f"{SAVE_DIR}/batch01_metadata.csv", index=False)
print(f"\n完成！已存到 Drive: batch01_metadata.csv")
print(f"其中有PMC开放全文: {df['has_fulltext'].sum()} 篇")
df[["pmid","year","has_fulltext","title"]].head(10)

已抓取 20 / 50
已抓取 40 / 50
已抓取 50 / 50

完成！已存到 Drive: batch01_metadata.csv
其中有PMC开放全文: 22 篇


,pmid,year,has_fulltext,title
0,39891270,2025,True,Lemon-derived nanoparticle-functionalized hydr...
1,37664860,2023,True,Current status and progress in research on dre...
2,37167893,2023,False,Snail-inspired AFG/GelMA hydrogel accelerates ...
3,40118421,2025,False,Hydrogel-based dressing for wound healing: A s...
4,30662554,2019,True,Engineering Bioactive Self-Healing Antibacteri...
5,40581219,2025,False,MSC-derived exosomes injectable hyaluronic aci...
6,37535449,2023,False,Ultrasound-Augmented Multienzyme-like Nanozyme...
7,35857459,2022,True,ECM-mimetic immunomodulatory hydrogel for meth...
8,38296937,2024,True,Hydrogel dressings with intrinsic antibiofilm ...
9,40050885,2025,True,Multifunctional hydrogel targeting senescence ...


In [5]:
import requests, re, time, json
import xml.etree.ElementTree as ET
import pandas as pd

df = pd.read_csv(f"{SAVE_DIR}/batch01_metadata.csv")
df_pmc = df[df["pmc"].notna() & (df["pmc"] != "")].copy()
print(f"有PMC全文的文献: {len(df_pmc)} 篇")

def parse_fulltext(xml_bytes):
    root = ET.fromstring(xml_bytes)
    t = root.find(".//article-title")
    title = "".join(t.itertext()) if t is not None else ""
    abstract = " ".join("".join(a.itertext()) for a in root.findall(".//abstract//p"))
    sections = []
    for sec in root.findall(".//body//sec"):
        st = sec.find("title")
        sec_title = "".join(st.itertext()) if st is not None else "(无标题)"
        paras = [re.sub(r"\s+", " ", "".join(p.itertext())).strip()
                 for p in sec.findall("p")]
        if paras:
            sections.append({"section": sec_title.strip(), "text": " ".join(paras)})
    return {"title": title.strip(), "abstract": abstract.strip(), "sections": sections}

papers, failed = [], []
for _, row in df_pmc.iterrows():
    try:
        url = f"https://www.ebi.ac.uk/europepmc/webservices/rest/{row['pmc']}/fullTextXML"
        r = requests.get(url, timeout=60)
        if r.status_code != 200:
            failed.append((row["pmid"], r.status_code)); continue
        doc = parse_fulltext(r.content)
        doc.update({"pmid": row["pmid"], "pmc": row["pmc"],
                    "doi": row["doi"] if pd.notna(row["doi"]) else "",
                    "year": row["year"], "journal": row["journal"]})
        papers.append(doc)
        print(f"✓ {row['pmid']}  {len(doc['sections'])}节  {doc['title'][:50]}...")
        time.sleep(0.3)
    except Exception as e:
        failed.append((row["pmid"], str(e)[:50]))

print(f"\n成功 {len(papers)} 篇，失败 {len(failed)} 篇 {failed if failed else ''}")

有PMC全文的文献: 22 篇
✓ 39891270  36节  Lemon-derived nanoparticle-functionalized hydrogel...
✓ 37664860  48节  Current status and progress in research on dressin...
✓ 30662554  8节  Engineering Bioactive Self-Healing Antibacterial E...
✓ 35857459  19节  ECM-mimetic immunomodulatory hydrogel for methicil...
✓ 38296937  35节  Hydrogel dressings with intrinsic antibiofilm and ...
✓ 40050885  36节  Multifunctional hydrogel targeting senescence to a...
✓ 38286650  34节  MiR‐17‐5p‐engineered sEVs Encapsulated in GelMA Hy...
✓ 38379700  31节  Versatile dopamine-functionalized hyaluronic acid-...
✓ 35898438  19节  Nanobiotechnology: Applications in Chronic Wound H...
✓ 40788421  54节  Poloxamer-based hydrogel with EGCG and rhEGF for d...
✓ 38140076  21节  Smart Responsive and Controlled-Release Hydrogels ...
✓ 38493156  35节  Anti-inflammatory and anti-oxidative electrospun n...
✓ 40942222  31节  Progressive Hydrogel Applications in Diabetic Foot...
✓ 36714604  9节  Diverse nanocomposites as a potential dressing

In [6]:
with open(f"{SAVE_DIR}/batch01_fulltext.jsonl", "w", encoding="utf-8") as f:
    for p in papers:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# 体检：每篇有多少字、覆盖哪些章节类型
stats = []
for p in papers:
    full_len = len(p["abstract"]) + sum(len(s["text"]) for s in p["sections"])
    sec_names = [s["section"].lower() for s in p["sections"]]
    stats.append({
        "pmid": p["pmid"], "字符数": full_len,
        "有方法": any("method" in s or "material" in s for s in sec_names),
        "有结果": any("result" in s for s in sec_names),
    })
stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))
print(f"\n已存到 Drive: batch01_fulltext.jsonl ({len(papers)}篇)")

    pmid    字符数   有方法   有结果
39891270  55784  True False
37664860 106293  True False
30662554  30091 False False
35857459  40054  True False
38296937  81169  True  True
40050885  48077  True False
38286650  55574 False False
38379700  54279  True False
35898438  48460  True False
40788421  80038  True False
38140076  34987 False False
38493156  59074  True False
40942222 102250  True False
36714604  22857 False False
40575654  45430 False False
35200508  44214 False False
37998956  48136 False False
33397416  43223  True False
41509512  94377 False False
36518979  44364  True False

已存到 Drive: batch01_fulltext.jsonl (20篇)


In [8]:
from google.colab import userdata
from openai import OpenAI
import json, time, re

client = OpenAI(
    api_key=userdata.get('BAILIAN_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)
MODEL = "qwen-plus"   # 抽取任务性价比最高；想更省用 qwen-turbo，想更准用 qwen-max

SCHEMA_PROMPT = """你是创面修复与生物材料领域的文献信息抽取助手。请从给定论文中抽取结构化信息。

【硬性规则】
1. 只抽取论文中明确写到的内容，严禁推测、补充、编造
2. 论文没提到的字段填 "NA"
3. 每个字段必须附上evidence：从原文复制一句支持该抽取的英文原句（逐字复制，不得改写）
4. 只输出JSON，不要输出任何其他文字

【抽取schema】
{
  "material_matrix": "水凝胶/敷料的基体材料，如 chitosan, GelMA, hyaluronic acid；无则NA",
  "crosslink_or_structure": "交联方式或结构特征，如 dynamic boronate ester bonds, nanofiber, microneedle；无则NA",
  "cargo": "负载物，如 growth factor, exosome, antibacterial peptide, metal ions；无则NA",
  "response_mechanism": "智能响应机制，如 pH-responsive, ROS-responsive, enzyme-responsive；非响应型填NA",
  "wound_model": "实验模型，如 db/db mice, STZ-induced diabetic rats, clinical patients, in vitro only",
  "pathogens": "涉及病原菌，如 S. aureus, P. aeruginosa, biofilm；无则NA",
  "outcomes": "性能结果数组，如 wound closure rate 95% at day 14, enhanced angiogenesis (CD31+)",
  "evidence_level": "in vitro / animal / clinical / review 四选一",
  "key_innovation": "作者声称的核心创新点，一句话",
  "evidence": {
    "material_matrix": "原文句子",
    "cargo": "原文句子",
    "response_mechanism": "原文句子",
    "outcomes": "原文句子"
  }
}
"""
print("schema就绪，模型:", MODEL)

schema就绪，模型: qwen-plus


In [11]:
def extract_paper(paper):
    # 摘要 + 正文前12000字符，控制成本又保住核心信息
    content = paper["abstract"] + "\n\n"
    for s in paper["sections"]:
        content += f"## {s['section']}\n{s['text']}\n"
        if len(content) > 12000:
            break
    content = content[:12000]

    resp = client.chat.completions.create(
        model=MODEL,                       # ← 就改了这一行
        temperature=0,
        messages=[
            {"role": "system", "content": SCHEMA_PROMPT},
            {"role": "user", "content": f"论文标题：{paper['title']}\n\n论文内容：\n{content}"}
        ],
    )
    text = resp.choices[0].message.content
    match = re.search(r'\{.*\}', text, re.DOTALL)  # 防止模型在JSON外多说了话
    return json.loads(match.group())

results, errors = [], []
for i, p in enumerate(papers):
    try:
        info = extract_paper(p)
        info["pmid"] = p["pmid"]      # PMID由程序挂上，不让LLM写——从根上杜绝假引用
        info["title"] = p["title"]
        info["doi"] = p["doi"]
        info["year"] = p["year"]
        results.append(info)
        print(f"✓ {i+1}/{len(papers)} {p['pmid']} | 材料: {str(info.get('material_matrix'))[:40]}")
        time.sleep(0.5)
    except Exception as e:
        errors.append((p["pmid"], str(e)[:60]))
        print(f"✗ {p['pmid']} 失败: {str(e)[:60]}")

with open(f"{SAVE_DIR}/batch01_extracted.jsonl", "w", encoding="utf-8") as f:
    for r_ in results:
        f.write(json.dumps(r_, ensure_ascii=False) + "\n")
print(f"\n成功 {len(results)} 篇，失败 {len(errors)} 篇，已存 batch01_extracted.jsonl")

✓ 1/20 39891270 | 材料: Gelatin Methacryloyl (GelMA) and Dialdeh
✓ 2/20 37664860 | 材料: NA
✓ 3/20 30662554 | 材料: Pluronic F127, oxidative hyaluronic acid
✓ 4/20 35857459 | 材料: glycopeptide hybrid hydrogel
✓ 5/20 38296937 | 材料: polyethylene glycol (PEG)
✓ 6/20 40050885 | 材料: gelatine methacryloyl (GelMA), sodium al
✓ 7/20 38286650 | 材料: GelMA
✓ 8/20 38379700 | 材料: hyaluronic acid, recombinant human colla
✓ 9/20 35898438 | 材料: NA
✓ 10/20 40788421 | 材料: hyaluronic acid, poloxamer 407, pectin
✓ 11/20 38140076 | 材料: NA
✓ 12/20 38493156 | 材料: chitosan, polycaprolactone/gelatin
✓ 13/20 40942222 | 材料: hyaluronic acid, chitosan, cellulose der
✓ 14/20 36714604 | 材料: NA
✓ 15/20 40575654 | 材料: Chlorella extracts, gelatin, polyethylen
✓ 16/20 35200508 | 材料: NA
✓ 17/20 37998956 | 材料: NA
✓ 18/20 33397416 | 材料: NA
✓ 19/20 41509512 | 材料: hydrogels
✓ 20/20 36518979 | 材料: NA

成功 20 篇，失败 0 篇，已存 batch01_extracted.jsonl


In [13]:
import pandas as pd, json
from collections import Counter

rows = []
for r_ in results:
    rows.append({
        "pmid": r_["pmid"],
        "year": r_["year"],
        "材料基体": r_.get("material_matrix"),
        "结构/交联": str(r_.get("crosslink_or_structure"))[:60],
        "负载物": str(r_.get("cargo"))[:60],
        "响应机制": r_.get("response_mechanism"),
        "模型": str(r_.get("wound_model"))[:40],
        "病原菌": str(r_.get("pathogens"))[:40],
        "证据等级": r_.get("evidence_level"),
    })
table = pd.DataFrame(rows)
table.to_csv(f"{SAVE_DIR}/batch01_table.csv", index=False)

print("=== 证据等级分布 ===")
print(table["证据等级"].value_counts())
print("\n=== 响应机制分布 ===")
print(table[table["响应机制"]!="NA"]["响应机制"].value_counts())
print("\n=== NA（综述类）的篇目 ===")
print(table[table["材料基体"]=="NA"][["pmid","证据等级"]].to_string(index=False))
print(f"\n总表已存 batch01_table.csv")
table

=== 证据等级分布 ===
证据等级
animal      10
review       9
in vitro     1
Name: count, dtype: int64

=== 响应机制分布 ===
响应机制
pH-responsive                                                                                                     1
pH- and MMP-2/9–responsive                                                                                        1
thermosensitive                                                                                                   1
pH-responsive, ROS-responsive, enzyme-responsive, glucose-responsive, light-responsive, electricity-responsive    1
pH-responsive, ROS-responsive, thermos-responsive                                                                 1
temperature-responsive                                                                                            1
Name: count, dtype: int64

=== NA（综述类）的篇目 ===
    pmid   证据等级
37664860 review
35898438 review
38140076 review
36714604 animal
35200508 review
37998956 review
33397416 review
36518979 review

总

,pmid,year,材料基体,结构/交联,负载物,响应机制,模型,病原菌,证据等级
0,39891270,2025,Gelatin Methacryloyl (GelMA) and Dialdehyde St...,photocrosslinkable,Lemon exosomes,NA,type 1 diabetic rat model,NA,animal
1,37664860,2023,NA,NA,NA,NA,NA,S. aureus (MSSA—methicillin-susceptible,review
2,30662554,2019,"Pluronic F127, oxidative hyaluronic acid (OHA)...",reversible Schiff base reaction between OHA an...,adipose-derived mesenchymal stem cells exosome...,pH-responsive,in vivo full-thickness diabetic wound,NA,animal
3,35857459,2022,glycopeptide hybrid hydrogel,porous structure cross-linked by hierarchical ...,antibacterial peptide (ILPWKWPWWPWRR),pH- and MMP-2/9–responsive,MRSA–infected full-thickness diabetic an,methicillin-resistant Staphylococcus aur,animal
4,38296937,2024,polyethylene glycol (PEG),crosslinked network with thiol-maleimide chemi...,cationic polyimidazolium (PIM) and N-acetylcys...,NA,murine diabetic wound model,methicillin-resistant Staphylococcus aur,animal
5,40050885,2025,"gelatine methacryloyl (GelMA), sodium alginate",UV crosslinking,"Panax notoginseng saponins (PNS), insulin-like...",NA,rat model,NA,animal
6,38286650,2024,GelMA,photo-crosslinked hydrogel,miR-17-5p-engineered small extracellular vesic...,NA,diabetic wounds,NA,animal
7,38379700,2024,"hyaluronic acid, recombinant human collagen ty...",oxidative coupling of the catechol group using...,NA,NA,diabetic rat,NA,animal
8,35898438,2022,NA,NA,NA,NA,in vitro only,NA,review
9,40788421,2025,"hyaluronic acid, poloxamer 407, pectin","EDC/NHS crosslinking, porous nanoarchitecture",EGCG and rhEGF,thermosensitive,in vitro only,"Escherichia coli, Staphylococcus aureus",in vitro


In [14]:
# 给LLM一张紧凑的"文献地图"，让它系统性地找空白
corpus_brief = []
for r_ in results:
    if r_.get("evidence_level") == "review":
        continue   # 综述不进Gap原料，只用原始研究
    corpus_brief.append({
        "pmid": r_["pmid"],
        "材料": r_.get("material_matrix"),
        "结构": str(r_.get("crosslink_or_structure"))[:80],
        "负载": str(r_.get("cargo"))[:80],
        "响应": r_.get("response_mechanism"),
        "模型": str(r_.get("wound_model"))[:50],
        "创新点": str(r_.get("key_innovation"))[:150],
    })

valid_pmids = {c["pmid"] for c in corpus_brief}   # 白名单：只允许引用语料库里的PMID

GAP_PROMPT = """你是创面修复领域的资深研究员。下面是一个文献语料库的结构化抽取结果（慢性难愈创面×智能敷料方向）。
请系统性地扫描这批文献，找出真实存在的研究空白（Research Gap）。

【找Gap的角度】
1. 组合空白：材料A的某机制在急性创面/其他场景验证过，但没人用于慢性创面
2. 机制空白：某响应机制（pH/ROS/酶）只在少数材料体系实现过，可否迁移
3. 负载空白：某有效成分（外泌体/抗菌肽/金属离子）与某基体的组合未见报道
4. 转化空白：只有体外数据、从未进动物模型的有潜力体系
5. 病原空白：针对生物膜/特定耐药菌的智能敷料缺口

【硬性规则】
1. 每条Gap必须引用语料库中存在的PMID作为支撑，严禁编造PMID
2. 输出8-12条，JSON数组，格式：
[{"gap_id": 1, "description": "Gap中文描述", "type": "上述5类之一",
  "supporting_pmids": ["PMID1","PMID2"], "rationale": "为什么这是空白、为什么值得做",
  "feasibility": "高/中/低", "novelty_query": "用于查重的PubMed英文检索式"}]
3. 只输出JSON"""

resp = client.chat.completions.create(
    model=MODEL, temperature=0.3,
    messages=[
        {"role": "system", "content": GAP_PROMPT},
        {"role": "user", "content": json.dumps(corpus_brief, ensure_ascii=False)}
    ],
)
text = resp.choices[0].message.content
gaps = json.loads(re.search(r'\[.*\]', text, re.DOTALL).group())

# 引用真实性校验：PMID必须在白名单里，不在的整条Gap打回
clean_gaps = []
for g in gaps:
    cited = [p for p in g.get("supporting_pmids", []) if str(p) in valid_pmids]
    if cited:
        g["supporting_pmids"] = cited
        clean_gaps.append(g)
    else:
        print(f"⚠ 打回一条引用无效的Gap: {g.get('description','')[:40]}")

with open(f"{SAVE_DIR}/batch01_gaps.json", "w", encoding="utf-8") as f:
    json.dump(clean_gaps, f, ensure_ascii=False, indent=2)

print(f"\n生成 {len(gaps)} 条Gap，校验通过 {len(clean_gaps)} 条\n")
for g in clean_gaps:
    print(f"[{g['gap_id']}] {g['type']} | {g['description'][:55]}")
    print(f"     支撑: {g['supporting_pmids']} | 可行性: {g['feasibility']}")

⚠ 打回一条引用无效的Gap: pH/ROS双响应型智能敷料在慢性难愈创面中尚未与外泌体负载体系联用——现有pH
⚠ 打回一条引用无效的Gap: 抗菌肽ILPWKWPWWPWRR（PMID35857459）仅与糖肽水凝胶联用，
⚠ 打回一条引用无效的Gap: miR-17-5p工程化sEVs（PMID38286650）仅用于GelMA单材
⚠ 打回一条引用无效的Gap: 热敏型水凝胶（PMID40788421, 40575654）全部限于体外或小鼠模
⚠ 打回一条引用无效的Gap: 针对铜绿假单胞菌（P. aeruginosa）生物膜的智能敷料完全缺失——语料库
⚠ 打回一条引用无效的Gap: 4-Octyl itaconate（OI）作为新型免疫代谢调节剂（PMID384
⚠ 打回一条引用无效的Gap: Panax notoginseng saponins（PNS）与IGF-1共载体
⚠ 打回一条引用无效的Gap: Chlorella提取物（PMID40575654）仅以物理自组装形式负载，未尝
⚠ 打回一条引用无效的Gap: 所有ROS响应体系（PMID36714604）均未明确关联特定ROS种类（如H2
⚠ 打回一条引用无效的Gap: rhEGF（PMID40788421）与EGCG共载体系仅验证体外活性，未进入糖

生成 10 条Gap，校验通过 0 条



In [15]:
# 修复：两边统一转成文字再比对
valid_pmids = {str(c["pmid"]) for c in corpus_brief}
print("语料库有效PMID池:", sorted(valid_pmids))

clean_gaps, dropped = [], []
for g in gaps:
    raw = [str(p) for p in g.get("supporting_pmids", [])]
    cited = [p for p in raw if p in valid_pmids]
    invalid = [p for p in raw if p not in valid_pmids]
    if cited:
        g["supporting_pmids"] = cited
        clean_gaps.append(g)
        if invalid:
            print(f"⚠ [{g['gap_id']}] 剔除了编造的PMID: {invalid}")
    else:
        dropped.append(g["description"][:40])
        print(f"✗ [{g['gap_id']}] 整条打回，引用的PMID全不在语料库: {raw}")

with open(f"{SAVE_DIR}/batch01_gaps.json", "w", encoding="utf-8") as f:
    json.dump(clean_gaps, f, ensure_ascii=False, indent=2)

print(f"\n校验通过 {len(clean_gaps)} / {len(gaps)} 条\n")
for g in clean_gaps:
    print(f"[{g['gap_id']}] {g['type']} | 可行性:{g['feasibility']}")
    print(f"   {g['description']}")
    print(f"   支撑文献: {g['supporting_pmids']}")
    print(f"   查重检索式: {g['novelty_query']}\n")

语料库有效PMID池: ['30662554', '35857459', '36714604', '38286650', '38296937', '38379700', '38493156', '39891270', '40050885', '40575654', '40788421']

校验通过 10 / 10 条

[1] 组合空白 | 可行性:高
   pH/ROS双响应型智能敷料在慢性难愈创面中尚未与外泌体负载体系联用——现有pH/ROS双响应载体（PMID36714604）未负载任何生物活性因子；而所有外泌体负载研究（PMID39891270、30662554、38286650）均无ROS响应设计，仅PMID30662554具pH响应但未整合ROS传感释放机制。
   支撑文献: ['36714604', '39891270', '30662554', '38286650']
   查重检索式: (("chronic wound" OR "diabetic ulcer") AND ("pH-responsive" AND "ROS-responsive" AND (exosome OR "extracellular vesicle"))) NOT ("acute wound" OR "burn")

[2] 负载空白 | 可行性:高
   抗菌肽ILPWKWPWWPWRR（PMID35857459）仅与糖肽水凝胶联用，尚未尝试负载于GelMA或HA等主流可光交联基质中——GelMA（PMID39891270, 38286650, 40050885）和HA（PMID30662554, 38379700, 40788421）均具优异生物相容性与可修饰性，但均未报道搭载该广谱抗MRSA肽。
   支撑文献: ['35857459', '39891270', '38286650', '40050885', '30662554', '38379700', '40788421']
   查重检索式: ("ILPWKWPWWPWRR" OR "antibacterial peptide") AND ("GelMA" OR "gelatin methacryloyl" OR "hyaluronic acid") AND ("chronic wound" OR "diab

In [16]:
def pubmed_search(query, retmax=10):
    r = requests.get(f"{BASE}/esearch.fcgi", params={
        "db": "pubmed", "term": query, "retmax": retmax, "retmode": "json"}, timeout=30)
    res = r.json()["esearchresult"]
    return int(res["count"]), res["idlist"]

def fetch_titles(ids):
    if not ids: return {}
    r = requests.get(f"{BASE}/esummary.fcgi", params={
        "db": "pubmed", "id": ",".join(ids), "retmode": "json"}, timeout=30)
    res = r.json()["result"]
    return {pid: res[pid]["title"] for pid in ids if pid in res}

JUDGE_PROMPT = """你是文献查重评审员。给定一条研究Gap主张和一批PubMed检索命中文献（PMID+标题），
判断每条命中文献是否已经做了Gap主张的事。
【规则】只根据标题判断，拿不准的标 borderline；只输出JSON：
{"hits": [{"pmid": "...", "verdict": "done相关已做/related沾边/irrelevant无关", "reason": "一句话"}],
 "overall": "A未见报道 / B沾边但未完全覆盖 / C基本已有人做",
 "comment": "一句话总结"}"""

novelty_log = []
for g in clean_gaps:
    count, ids = pubmed_search(g["novelty_query"])
    titles = fetch_titles(ids[:10])

    if count == 0:
        verdict, comment, hit_judgments = "A未见报道", "全库0命中", []
    else:
        resp = client.chat.completions.create(
            model=MODEL, temperature=0,
            messages=[
                {"role": "system", "content": JUDGE_PROMPT},
                {"role": "user", "content":
                 f"Gap主张：{g['description']}\n\n命中文献：\n" +
                 "\n".join(f"{pid}: {t}" for pid, t in titles.items())}
            ],
        )
        j = json.loads(re.search(r'\{.*\}', resp.choices[0].message.content, re.DOTALL).group())
        verdict, comment = j.get("overall", "?"), j.get("comment", "")
        hit_judgments = j.get("hits", [])

    novelty_log.append({
        "gap_id": g["gap_id"], "description": g["description"],
        "query": g["novelty_query"], "hit_count": count,
        "hits": hit_judgments, "verdict": verdict, "comment": comment,
    })
    print(f"[{g['gap_id']}] 命中{count:>3}篇 → {verdict} | {g['description'][:38]}")
    print(f"      {comment}")
    time.sleep(1)

with open(f"{SAVE_DIR}/batch01_novelty_log.json", "w", encoding="utf-8") as f:
    json.dump(novelty_log, f, ensure_ascii=False, indent=2)
print(f"\n查重日志已存 batch01_novelty_log.json（评委可复核每一条）")


[1] 命中  0篇 → A未见报道 | pH/ROS双响应型智能敷料在慢性难愈创面中尚未与外泌体负载体系联用——现有
      全库0命中
[2] 命中  0篇 → A未见报道 | 抗菌肽ILPWKWPWWPWRR（PMID35857459）仅与糖肽水凝胶联
      全库0命中
[3] 命中  0篇 → A未见报道 | miR-17-5p工程化sEVs（PMID38286650）仅用于GelMA
      全库0命中
[4] 命中  0篇 → A未见报道 | 热敏型水凝胶（PMID40788421, 40575654）全部限于体外或小
      全库0命中
[5] 命中 10篇 → B沾边但未完全覆盖 | 针对铜绿假单胞菌（P. aeruginosa）生物膜的智能敷料完全缺失——语
      仅PMID36902169明确针对Pseudomonas sp.生物膜与伤口敷料（细菌纤维素基），但未强调‘智能响应’（如pH/酶/ROS等动态调控）；其余文献或病原体不符、或场景不符、或缺乏P. aeruginosa特异性及慢性创面验证，故Gap核心——‘针对P. aeruginosa生物膜的智能敷料’仍基本未被满足。
[6] 命中  1篇 → C基本已有人做 | 4-Octyl itaconate（OI）作为新型免疫代谢调节剂（PMID3
      该文献已实现OI在可注射水凝胶中的应用，且明确强调‘injectable hydrogel’，完全覆盖Gap所称‘未见于可注射水凝胶体系’这一断言。
[7] 命中  0篇 → A未见报道 | Panax notoginseng saponins（PNS）与IGF-1共
      全库0命中
[8] 命中  1篇 → C基本已有人做 | Chlorella提取物（PMID40575654）仅以物理自组装形式负载，
      虽未直接报道Chlorella提取物与DAS的Schiff base偶联，但DAS-Chlorella体系已在实际应用中构建，表明关键材料组合与化学基础已被验证。
[9] 命中  0篇 → A未见报道 | 所有ROS响应体系（PMID36714604）均未明确关联特定ROS种类（如
      全库0命中
[10] 命中  0篇 → A未见报道 | rhEGF（PMID40788421）与E

In [17]:
SELECTED_GAPS = [1, 2, 9]   # ← 想换就改这里

# 为每条Gap组装"证据包"：支撑文献的抽取结果＋摘要＋原文证据句
def build_evidence_bundle(gap):
    bundle = {"gap": gap["description"], "novelty": "全库查重未见报道",
              "papers": []}
    for pmid in gap["supporting_pmids"]:
        ext = next((r_ for r_ in results if str(r_["pmid"]) == str(pmid)), None)
        pap = next((p for p in papers if str(p["pmid"]) == str(pmid)), None)
        if ext and pap:
            bundle["papers"].append({
                "pmid": str(pmid), "title": pap["title"],
                "abstract": pap["abstract"][:1500],
                "extracted": {k: v for k, v in ext.items()
                              if k not in ("pmid","title","doi","year","evidence")},
                "evidence_quotes": ext.get("evidence", {}),
            })
    return bundle

HYPO_PROMPT = """你是创面修复领域的首席科学家。基于给定的研究空白（已全库查重确认未见报道）和支撑文献证据包，
生成一个可实验验证的研究假设。

【硬性规则】
1. evidence_chain中的每条quote必须逐字复制自证据包中提供的abstract或evidence_quotes，严禁自己造句子
2. 假设要具体：材料组成、负载物、响应机制、验证模型都要明确
3. 只输出JSON：
{
  "hypothesis": "假设陈述（中文，一段话说清：什么材料体系，通过什么机制，在什么模型上预期什么效果）",
  "mechanism_rationale": "机制推理链（为什么预期有效，逐步推导）",
  "experimental_design": {
    "materials_plan": "材料制备要点",
    "models": "体外+动物模型设计",
    "controls": "关键对照组",
    "endpoints": "主要/次要观察指标"
  },
  "evidence_chain": [
    {"claim": "推理中的某个事实依据", "pmid": "来源", "quote": "逐字原文"}
  ],
  "known_vs_new": {"already_known": "已有文献证明的部分", "our_increment": "本假设新增的部分"},
  "risks": "最可能失败的环节"
}"""

hypotheses = []
for g in clean_gaps:
    if g["gap_id"] not in SELECTED_GAPS:
        continue
    bundle = build_evidence_bundle(g)
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.4,
        messages=[
            {"role": "system", "content": HYPO_PROMPT},
            {"role": "user", "content": json.dumps(bundle, ensure_ascii=False)}
        ],
    )
    h = json.loads(re.search(r'\{.*\}', resp.choices[0].message.content, re.DOTALL).group())
    h["gap_id"] = g["gap_id"]
    hypotheses.append(h)
    print(f"✓ Gap[{g['gap_id']}] 假设已生成，证据链 {len(h.get('evidence_chain',[]))} 环")
    time.sleep(1)

✓ Gap[1] 假设已生成，证据链 3 环
✓ Gap[2] 假设已生成，证据链 4 环
✓ Gap[9] 假设已生成，证据链 3 环


In [18]:
def difflib_ok(quote_n, src, threshold=0.85):
    """引句与原文局部相似度兜底：滑动窗口比对"""
    words = quote_n.split()
    if len(words) < 5:
        return False
    src_words = src.split()
    win = len(words)
    best = 0
    for i in range(0, max(1, len(src_words) - win + 1), 3):
        seg = " ".join(src_words[i:i+win])
        r = difflib.SequenceMatcher(None, quote_n, seg).ratio()
        if r > best:
            best = r
        if best >= threshold:
            return True
    return False

import difflib

In [19]:
import unicodedata

def normalize(s):
    s = unicodedata.normalize("NFKC", s).lower()
    return re.sub(r"[^a-z0-9\u4e00-\u9fff]+", " ", s).strip()

# 建原文索引：pmid → 归一化全文
corpus_text = {}
for p in papers:
    full = p["title"] + " " + p["abstract"] + " " + \
           " ".join(s_["text"] for s_ in p["sections"])
    corpus_text[str(p["pmid"])] = normalize(full)

print("=== 证据链逐字核验 ===\n")
for h in hypotheses:
    ok, bad = 0, []
    for link in h.get("evidence_chain", []):
        quote_n = normalize(str(link.get("quote", "")))
        src = corpus_text.get(str(link.get("pmid")), "")
        # 逐字包含，或80%以上词序列匹配（容忍大小写/标点差异）
        if quote_n in src or \
           (quote_n and difflib_ok(quote_n, src)):
            ok += 1
        else:
            bad.append(link)
    h["verified_links"] = ok
    h["failed_links"] = bad
    status = "✅ 全部通过" if not bad else f"⚠ {len(bad)}环未通过已剔除"
    print(f"Gap[{h['gap_id']}] 证据链 {ok} 环通过 | {status}")

# 剔除无法验证的引用环，净化最终版本
for h in hypotheses:
    h["evidence_chain"] = [l for l in h.get("evidence_chain", [])
                           if l not in h["failed_links"]]
    del h["failed_links"]

with open(f"{SAVE_DIR}/batch01_hypotheses.json", "w", encoding="utf-8") as f:
    json.dump(hypotheses, f, ensure_ascii=False, indent=2)
print("\n已存 batch01_hypotheses.json（核验后的净版）")

=== 证据链逐字核验 ===

Gap[1] 证据链 2 环通过 | ⚠ 1环未通过已剔除
Gap[2] 证据链 4 环通过 | ✅ 全部通过
Gap[9] 证据链 2 环通过 | ⚠ 1环未通过已剔除

已存 batch01_hypotheses.json（核验后的净版）


In [20]:
for h in hypotheses:
    print("="*70)
    print(f"【Gap {h['gap_id']} 的研究假设】")
    print(f"\n◆ 假设陈述\n{h['hypothesis']}")
    print(f"\n◆ 机制推理\n{h['mechanism_rationale']}")
    print(f"\n◆ 实验设计")
    for k, v in h["experimental_design"].items():
        print(f"  · {k}: {v}")
    print(f"\n◆ 证据链（{len(h['evidence_chain'])}环，均已逐字核验）")
    for i, link in enumerate(h["evidence_chain"], 1):
        print(f"  [{i}] {link['claim']}")
        print(f"      来源 PMID:{link['pmid']}")
        print(f"      原文: \"{link['quote'][:120]}...\"")
    print(f"\n◆ 已知 vs 新增")
    print(f"  已知: {h['known_vs_new']['already_known']}")
    print(f"  新增: {h['known_vs_new']['our_increment']}")
    print(f"\n◆ 风险: {h['risks']}\n")

【Gap 1 的研究假设】

◆ 假设陈述
以氧化海藻酸钠（OHA）与聚-ε-L-lysine（EPL）通过席夫碱动态共价交联构建pH/ROS双响应水凝胶基质，负载柠檬来源外泌体（20 μg/mL），该体系在糖尿病大鼠全层皮肤创面模型中，可于病理性酸性微环境（pH ≈ 5.5–6.2）及高ROS水平（H₂O₂ ≥ 50 μM）双重刺激下协同断裂席夫碱键并氧化裂解苯硼酸酯键，实现外泌体的时空精准释放，从而显著促进M2型巨噬细胞极化、血管内皮细胞管腔形成及胶原有序沉积，较单pH响应对照组提升创面愈合速率≥40%（第7天）并减少瘢痕面积≥35%（第14天）。

◆ 机制推理
1) 慢性糖尿病创面微环境具有特征性低pH（5.5–6.2）与高ROS（H₂O₂达50–200 μM）双重病理信号；2) PMID30662554证实OHA/EPL席夫碱水凝胶可在弱酸环境下断裂释放外泌体：'The exosomes could be released under a weak acidic environment due to the broken of Schiff base bonds.'；3) PMID36714604明确报道'phenylboronic acid and polyvinyl alcohol also contained ROS and exhibited anti-inflammatory action'，且该体系已验证ROS响应性；4) PMID39891270证实柠檬外泌体'promoted diabetic wound healing by regulating macrophage polarization and promoting fibroblast and vascular endothelial cell proliferation'；5) 因此，将苯硼酸修饰的OHA引入FHE体系（替代部分OHA），构建OHA-PBA/EPL双动态网络，即可同时响应pH（席夫碱断裂）与ROS（苯硼酸酯氧化水解），实现外泌体在病灶核心区的双阈值触发释放，避免全身扩散与早期失活。

◆ 实验设计
  · materials_plan: 合成苯硼酸修饰氧化海藻酸钠（OHA-PBA）：NaIO₄氧化海藻酸钠得OHA，再与4-羧基苯硼酸经EDC/NHS偶联；与EPL按

In [21]:
from datetime import date

L = []  # report lines
L.append("# 慢性难愈创面智能敷料的文献驱动科学发现：调研报告\n")
L.append(f"生成日期：{date.today()} | 语料库：PubMed 20篇开放全文（2019-2026）\n")

L.append("\n## 1. 科学问题\n")
L.append("慢性难愈创面（糖尿病足溃疡、静脉性溃疡、压疮）的愈合停滞与微环境异常"
         "（高ROS、蛋白酶失衡、生物膜感染、巨噬细胞极化障碍）密切相关。"
         "智能响应型敷料能感知微环境并按需释药，但材料体系、响应机制与生物活性负载"
         "之间的组合空间远未被系统探索。本研究构建文献驱动的发现型Agent，"
         "从公开文献中自动识别该领域的研究空白并生成可验证假设。\n")

L.append("\n## 2. 方法：发现管线\n")
L.append("检索（PubMed E-utilities）→ 全文结构化解析（PMC XML）→ "
         "LLM实体抽取（9字段schema，逐字段附原文证据句）→ Gap系统扫描 → "
         "全库新颖性查重（检索式留档）→ 假设生成 → 证据链逐字核验。"
         "LLM仅参与内容生成；PMID挂载、引用校验、查重计数均由确定性程序执行。\n")

L.append("\n## 3. 文献抽取总表\n")
L.append(f"原始研究 {len([r_ for r_ in results if r_.get('evidence_level')!='review'])} 篇，"
         f"综述 {len([r_ for r_ in results if r_.get('evidence_level')=='review'])} 篇。"
         "完整表格见 batch01_table.csv。\n")

L.append("\n## 4. Gap清单与新颖性查重\n")
L.append("| # | 类型 | Gap | 查重命中 | 判定 |")
L.append("|---|------|-----|---------|------|")
for g, n in zip(clean_gaps, novelty_log):
    L.append(f"| {g['gap_id']} | {g['type']} | {g['description'][:60]}… "
             f"| {n['hit_count']}篇 | {n['verdict']} |")
L.append("\n查重检索式与逐条判读记录见 batch01_novelty_log.json（可复核）。\n")

L.append("\n## 5. 有效发现（3条可验证假设）\n")
for h in hypotheses:
    L.append(f"\n### 发现 {h['gap_id']}\n")
    L.append(f"**假设**：{h['hypothesis']}\n")
    L.append(f"**机制推理**：{h['mechanism_rationale']}\n")
    L.append(f"**已知 vs 新增**：已知——{h['known_vs_new']['already_known']}；"
             f"本研究新增——{h['known_vs_new']['our_increment']}\n")
    L.append("**证据链**（逐字核验通过）：")
    for link in h["evidence_chain"]:
        L.append(f"- {link['claim']}（PMID:{link['pmid']}：\"{link['quote'][:100]}…\"）")
    L.append(f"\n**验证方案**：{h['experimental_design']['models']}；"
             f"对照：{h['experimental_design']['controls']}；"
             f"指标：{h['experimental_design']['endpoints']}\n")

L.append("\n## 6. 附录：数据与日志清单\n")
L.append("- batch01_metadata.csv：50篇检索元数据\n"
         "- batch01_fulltext.jsonl：20篇结构化全文\n"
         "- batch01_extracted.jsonl：20篇实体抽取（含证据句）\n"
         "- batch01_gaps.json：10条Gap（PMID白名单校验通过）\n"
         "- batch01_novelty_log.json：全库查重日志\n"
         "- batch01_hypotheses.json：3条假设（证据链已净化）\n")

report = "\n".join(L)
with open(f"{SAVE_DIR}/调研报告_v1.md", "w", encoding="utf-8") as f:
    f.write(report)
print(f"报告已生成：调研报告_v1.md（{len(report)}字符）")

报告已生成：调研报告_v1.md（7609字符）


In [22]:
# 只对Gap 9重生成：强制负载物为邻苯二酚类抗氧化剂
g9 = next(g for g in clean_gaps if g["gap_id"] == 9)
bundle = build_evidence_bundle(g9)
constraint = ("\n【额外约束】负载物必须是含邻苯二酚（catechol）结构的抗氧化剂"
              "（如多巴胺dopamine、没食子酸gallic acid、原儿茶酸protocatechuic acid），"
              "确保其能与苯硼酸形成硼酸酯键，化学上必须成立。")

resp = client.chat.completions.create(
    model=MODEL, temperature=0.4,
    messages=[
        {"role": "system", "content": HYPO_PROMPT + constraint},
        {"role": "user", "content": json.dumps(bundle, ensure_ascii=False)}
    ],
)
h9 = json.loads(re.search(r'\{.*\}', resp.choices[0].message.content, re.DOTALL).group())
h9["gap_id"] = 9

# 重新核验证据链
ok, bad = 0, []
for link in h9.get("evidence_chain", []):
    quote_n = normalize(str(link.get("quote","")))
    src = corpus_text.get(str(link.get("pmid")), "")
    if quote_n in src or (quote_n and difflib_ok(quote_n, src)):
        ok += 1
    else:
        bad.append(link)
h9["evidence_chain"] = [l for l in h9.get("evidence_chain",[]) if l not in bad]
print(f"核验通过 {ok} 环，剔除 {len(bad)} 环")

hypotheses = [h if h["gap_id"] != 9 else h9 for h in hypotheses]
print(h9["hypothesis"])

核验通过 2 环，剔除 1 环
以苯硼酸修饰的明胶-透明质酸双网络水凝胶为基质，负载含邻苯二酚结构的原儿茶酸（protocatechuic acid），通过ONOO⁻特异性触发苯硼酸-邻苯二酚硼酸酯键断裂，实现原儿茶酸的靶向释放；在db/db小鼠全层皮肤糖尿病创面模型中，该体系可选择性清除ONOO⁻、抑制硝化应激、促进M2巨噬细胞极化及血管新生，从而显著加速创面愈合。


In [24]:
# 把发现1的图注证据换成摘要中的实质结论句
h1 = next(h for h in hypotheses if h["gap_id"] == 1)
h1["evidence_chain"][1] = {
    "claim": "柠檬外泌体通过调控巨噬细胞极化及促进内皮细胞与成纤维细胞增殖迁移促进糖尿病伤口愈合",
    "pmid": "39891270",
    "quote": "promoted diabetic wound healing by regulating macrophage polarization and promoting fibroblast and vascular endothelial cell proliferation"
}
# 验证新引句确实在原文里
q = normalize(h1["evidence_chain"][1]["quote"])
print("验证:", "✅ 通过" if q in corpus_text["39891270"] or difflib_ok(q, corpus_text["39891270"]) else "❌ 未找到")

# 重新存盘并重跑格子⑩'生成报告
with open(f"{SAVE_DIR}/batch01_hypotheses.json", "w", encoding="utf-8") as f:
    json.dump(hypotheses, f, ensure_ascii=False, indent=2)

验证: ✅ 通过


In [25]:
def fmt_items(x, prefix="  - "):
    """把list/dict格式的字段排成人类可读列表"""
    if isinstance(x, list):
        return "\n".join(prefix + str(i) for i in x)
    if isinstance(x, dict):
        return "\n".join(f"  · **{k}**：\n" + "\n".join("    - " + str(i) for i in v)
                         if isinstance(v, list) else f"  · **{k}**：{v}"
                         for k, v in x.items())
    return str(x)

from datetime import date
L = []
L.append("# 慢性难愈创面智能敷料的文献驱动科学发现：调研报告\n")
L.append(f"> 生成日期：{date.today()}｜语料库：PubMed 20篇开放全文（2019–2026）")
L.append("> 本报告由文献发现Agent自动生成并经人工审核；仅用于科学研究，不构成任何临床诊疗建议。\n")

# 1-2节沿用
L.append("\n## 1. 科学问题\n")
L.append("慢性难愈创面（糖尿病足溃疡、静脉性溃疡、压疮）的愈合停滞与微环境异常"
         "（高ROS、蛋白酶失衡、生物膜感染、巨噬细胞极化障碍）密切相关。智能响应型敷料"
         "能感知微环境并按需释药，但材料体系、响应机制与生物活性负载之间的组合空间"
         "远未被系统探索。本研究构建文献驱动的发现型Agent，从公开文献中自动识别研究空白"
         "并生成可验证假设。\n")
L.append("\n## 2. 方法：发现管线\n")
L.append("检索（PubMed E-utilities）→ 全文结构化解析（PMC XML）→ LLM实体抽取"
         "（9字段schema，逐字段附原文证据句）→ Gap系统扫描 → 全库新颖性查重（检索式留档）"
         "→ 假设生成 → 证据链逐字核验。LLM仅参与内容生成；PMID挂载、引用校验、查重计数"
         "均由确定性程序执行。综述类文献仅作背景，不进入Gap生成原料。\n")

# 新增：文献交叉引用矩阵（对应"Gap清单+文献交叉引用"要求）
L.append("\n## 3. 文献交叉引用矩阵（材料 × 响应机制）\n")
mat = {}
for r_ in results:
    if r_.get("evidence_level") == "review":
        continue
    mat_key = (str(r_.get("material_matrix"))[:30], str(r_.get("response_mechanism")))
    mat.setdefault(mat_key, []).append(str(r_["pmid"]))
L.append("| 材料基体 | 响应机制 | 文献 |")
L.append("|---------|---------|------|")
for (m, mech), pms in sorted(mat.items()):
    L.append(f"| {m} | {mech} | {', '.join(pms)} |")

# 4节Gap表沿用（略，复制原版代码）…
L.append("\n## 4. Gap清单与新颖性查重\n")
L.append("| # | 类型 | Gap | 查重命中 | 判定 |")
L.append("|---|------|-----|---------|------|")
for g, n in zip(clean_gaps, novelty_log):
    L.append(f"| {g['gap_id']} | {g['type']} | {g['description'][:60]}… "
             f"| {n['hit_count']}篇 | {n['verdict']} |")
L.append("\n查重检索式与逐条判读记录见 batch01_novelty_log.json（可复核）。\n")

# 5节：假设，字段全部用fmt_items排版
L.append("\n## 5. 有效发现（3条可验证假设）\n")
for h in hypotheses:
    ed = h["experimental_design"]
    L.append(f"\n### 发现 {h['gap_id']}\n")
    L.append(f"**假设**：{h['hypothesis']}\n")
    L.append(f"**机制推理**：{h['mechanism_rationale']}\n")
    L.append(f"**已知 vs 新增**：\n- 已知：{h['known_vs_new']['already_known']}"
             f"\n- 本研究新增：{h['known_vs_new']['our_increment']}\n")
    L.append("**证据链**（逐字核验通过）：")
    for link in h["evidence_chain"]:
        L.append(f"- {link['claim']}（PMID:{link['pmid']}：\"{link['quote'][:100]}…\"）")
    L.append(f"\n**验证方案**：\n- 模型：{ed['models']}\n- 对照组：\n"
             + fmt_items(ed['controls'], "  - ")
             + f"\n- 观察指标：\n" + fmt_items(ed['endpoints'])
             + "\n> 注：上述量化指标为假设性预期目标，非既有结论。\n")

L.append("\n## 6. 附录：数据与日志清单\n")
L.append("- batch01_metadata.csv：50篇检索元数据\n- batch01_fulltext.jsonl：20篇结构化全文\n"
         "- batch01_extracted.jsonl：20篇实体抽取（含证据句）\n- batch01_gaps.json：10条Gap（PMID白名单校验通过）\n"
         "- batch01_novelty_log.json：全库查重日志\n- batch01_hypotheses.json：3条假设（证据链已净化）\n")

report = "\n".join(L)
with open(f"{SAVE_DIR}/调研报告_v2.md", "w", encoding="utf-8") as f:
    f.write(report)
print(f"报告已生成：调研报告_v2.md（{len(report)}字符）")

报告已生成：调研报告_v2.md（8607字符）


In [12]:
import json
papers = []
with open(f"{SAVE_DIR}/batch01_fulltext.jsonl", encoding="utf-8") as f:
    for line in f:
        papers.append(json.loads(line))
print(len(papers), "篇已加载")

20 篇已加载


In [27]:
MAIN_QUERY = (
    '("chronic wound"[Title/Abstract] OR "diabetic foot ulcer"[Title/Abstract] '
    'OR "non-healing wound"[Title/Abstract] OR "nonhealing wound"[Title/Abstract]) '
    'AND (hydrogel[Title/Abstract] OR "smart dressing"[Title/Abstract] '
    'OR scaffold[Title/Abstract] OR nanofiber[Title/Abstract]) '
    'AND (angiogenesis[Title/Abstract] OR antibacterial[Title/Abstract] '
    'OR "wound healing"[Title/Abstract]) '
    'AND ("2019/01/01"[Date - Publication] : "2026/12/31"[Date - Publication])'
)
SUP_QUERIES = {
    "ONOO⁻补充": '("peroxynitrite" OR "nitrosative stress" OR "nitrotyrosine") AND ("diabetic wound" OR "diabetic ulcer" OR "diabetic foot" OR "impaired wound healing")',
    "邻苯二酚补充": '("protocatechuic acid" OR "gallic acid" OR dopamine) AND ("wound healing" OR hydrogel) AND (antioxidant OR "diabetic wound")',
}

pool = set()
for start in (0, 100):   # 主检索翻两页，取200篇
    r = requests.get(f"{BASE}/esearch.fcgi", params={
        "db": "pubmed", "term": MAIN_QUERY, "retmax": 100, "retstart": start,
        "retmode": "json", "sort": "relevance", "email": EMAIL}, timeout=30)
    pool.update(r.json()["esearchresult"]["idlist"])
    time.sleep(0.4)
for name, q in SUP_QUERIES.items():
    r = requests.get(f"{BASE}/esearch.fcgi", params={
        "db": "pubmed", "term": q, "retmax": 15, "retmode": "json",
        "sort": "relevance", "email": EMAIL}, timeout=30)
    pool.update(r.json()["esearchresult"]["idlist"])
    time.sleep(0.4)

existing = {str(p["pmid"]) for p in papers}
new_pmids = sorted(pool - existing)
print(f"池内共{len(pool)}篇，已有{len(existing & pool)}篇，新增候选{len(new_pmids)}篇")

池内共226篇，已有20篇，新增候选206篇


In [29]:
def fetch_batch_meta(id_list):
    r = requests.get(f"{BASE}/efetch.fcgi", params={
        "db": "pubmed", "id": ",".join(id_list), "retmode": "xml", "email": EMAIL}, timeout=60)
    root = ET.fromstring(r.content)
    recs = []
    for art in root.findall(".//PubmedArticle"):
        t = art.find(".//ArticleTitle")
        recs.append({
            "pmid": art.findtext(".//PMID"),
            "title": "".join(t.itertext()) if t is not None else "",
            "journal": art.findtext(".//Journal/Title", ""),
            "year": art.findtext(".//JournalIssue/PubDate/Year") or "",
            "doi": next((a.text for a in art.findall(".//ArticleId") if a.get("IdType")=="doi"), ""),
            "pmc": next((a.text for a in art.findall(".//ArticleId") if a.get("IdType")=="pmc"), ""),
            "abstract": " ".join("".join(a.itertext()) for a in art.findall(".//Abstract/AbstractText")),
        })
    return recs

new_meta = []
for i in range(0, len(new_pmids), 20):
    new_meta += fetch_batch_meta(new_pmids[i:i+20])
    time.sleep(0.4)
new_df = pd.DataFrame(new_meta)
new_pmc = new_df[new_df["pmc"] != ""]
print(f"新增元数据{len(new_df)}篇，其中有PMC全文{len(new_pmc)}篇")
new_df.to_csv(f"{SAVE_DIR}/batch02_metadata.csv", index=False)

新增元数据206篇，其中有PMC全文70篇


In [30]:
def parse_fulltext(xml_bytes):
    root = ET.fromstring(xml_bytes)
    t = root.find(".//article-title")
    sections = []
    for sec in root.findall(".//body//sec"):
        st = sec.find("title")
        paras = [re.sub(r"\s+", " ", "".join(p.itertext())).strip() for p in sec.findall("p")]
        if paras:
            sections.append({"section": ("".join(st.itertext()) if st is not None else "(无标题)").strip(),
                             "text": " ".join(paras)})
    return {"title": ("".join(t.itertext()) if t is not None else "").strip(),
            "abstract": " ".join("".join(a.itertext()) for a in root.findall(".//abstract//p")).strip(),
            "sections": sections}

papers_new, failed = [], []
for _, row in new_pmc.iterrows():
    try:
        r = requests.get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/{row['pmc']}/fullTextXML", timeout=60)
        if r.status_code != 200:
            failed.append(row["pmid"]); continue
        doc = parse_fulltext(r.content)
        doc.update({"pmid": str(row["pmid"]), "pmc": row["pmc"], "doi": row["doi"],
                    "year": row["year"], "journal": row["journal"]})
        papers_new.append(doc)
        if len(papers_new) % 10 == 0:
            print(f"已抓 {len(papers_new)} 篇...")
        time.sleep(0.3)
    except Exception:
        failed.append(row["pmid"])

with open(f"{SAVE_DIR}/batch02_fulltext.jsonl", "w", encoding="utf-8") as f:
    for p in papers_new:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"新增全文{len(papers_new)}篇，失败{len(failed)}篇")

已抓 10 篇...
已抓 20 篇...
已抓 30 篇...
已抓 40 篇...
已抓 50 篇...
新增全文57篇，失败13篇


In [31]:
results_new, errors_new = [], []
for i, p in enumerate(papers_new):
    try:
        info = extract_paper(p)
        info.update({"pmid": p["pmid"], "title": p["title"], "doi": p["doi"], "year": p["year"]})
        results_new.append(info)
        print(f"✓ {i+1}/{len(papers_new)} {p['pmid']} | {str(info.get('material_matrix'))[:35]}")
        time.sleep(0.5)
    except Exception as e:
        errors_new.append(p["pmid"])
        print(f"✗ {p['pmid']}: {str(e)[:50]}")

with open(f"{SAVE_DIR}/batch02_extracted.jsonl", "w", encoding="utf-8") as f:
    for r_ in results_new:
        f.write(json.dumps(r_, ensure_ascii=False) + "\n")
print(f"\n新增抽取{len(results_new)}篇，失败{len(errors_new)}篇")

✓ 1/57 20566667 | NA
✓ 2/57 31083588 | alginate
✓ 3/57 31861794 | chitosan
✓ 4/57 32440554 | hydrogel
✓ 5/57 32900696 | collagen-glycosaminoglycan scaffold
✓ 6/57 34859323 | NA
✓ 7/57 34998407 | Gelatin methacryloyl (GelMA) and Po
✓ 8/57 35909814 | chitosan, chitosan-polyethylene gly
✓ 9/57 36234898 | 4-arm acrylated polyethylene glycol
✓ 10/57 36235942 | NA
✓ 11/57 36582352 | SilMA/HAMA
✓ 12/57 36761188 | NA
✓ 13/57 36811095 | GelMA and gelatin
✓ 14/57 36950149 | gelatin modified by dopamine (Gel-D
✓ 15/57 37038409 | polyvinyl alcohol (PVA)
✓ 16/57 37076933 | functionalized sodium alginate (FSA
✓ 17/57 37420287 | GelMA
✓ 18/57 37511198 | sodium alginate
✓ 19/57 38030563 | hyaluronic acid
✓ 20/57 38089434 | Acellular Dermal Matrix (ADM) and G
✓ 21/57 38784443 | Pluronic F-127 hydrogel
✓ 22/57 38797862 | Carboxymethyl Chitosan (CMCS), Dext
✓ 23/57 39210966 | oxidized hyaluronic acid (OHA) and 
✓ 24/57 39280109 | methacrylate gelatin (GelMa) and Po
✓ 25/57 39333183 | Bletilla striata pol

In [32]:
papers_all = papers + papers_new
results_all = results + results_new
print(f"=== 扩库完成 ===")
print(f"全文库: {len(papers)} → {len(papers_all)} 篇")
print(f"抽取库: {len(results)} → {len(results_all)} 篇")
print(f"原始研究: {len([r for r in results_all if r.get('evidence_level')!='review'])} 篇")

=== 扩库完成 ===
全文库: 20 → 77 篇
抽取库: 20 → 77 篇
原始研究: 56 篇


In [34]:
# 1) 建新的文献地图（用合并后的results_all，综述排除）
corpus_brief = []
for r_ in results_all:
    if r_.get("evidence_level") == "review":
        continue
    corpus_brief.append({
        "pmid": str(r_["pmid"]),
        "材料": str(r_.get("material_matrix"))[:60],
        "结构": str(r_.get("crosslink_or_structure"))[:60],
        "负载": str(r_.get("cargo"))[:60],
        "响应": r_.get("response_mechanism"),
        "模型": str(r_.get("wound_model"))[:40],
        "创新点": str(r_.get("key_innovation"))[:120],
    })
valid_pmids = {c["pmid"] for c in corpus_brief}   # 这次统一是字符串，不会再误杀
print(f"Gap原料: {len(corpus_brief)}篇原始研究")

# 2) 重新生成Gap（GAP_PROMPT沿用，不确定在不在内存就把旧格子⑤的prompt重跑一遍）
resp = client.chat.completions.create(
    model=MODEL, temperature=0.3,
    messages=[
        {"role": "system", "content": GAP_PROMPT},
        {"role": "user", "content": json.dumps(corpus_brief, ensure_ascii=False)}
    ],
)
gaps_v2 = json.loads(re.search(r'\[.*\]', resp.choices[0].message.content, re.DOTALL).group())

# 3) 白名单校验（字符串比对版）
clean_gaps = []
for g in gaps_v2:
    cited = [str(p) for p in g.get("supporting_pmids", []) if str(p) in valid_pmids]
    if cited:
        g["supporting_pmids"] = cited
        clean_gaps.append(g)
    else:
        print(f"⚠ 打回: {g.get('description','')[:40]}")
print(f"\n生成{len(gaps_v2)}条，通过{len(clean_gaps)}条")

# 4) 全库查重（JUDGE_PROMPT和pubmed_search/fetch_titles沿用旧格子⑥）
novelty_log = []
for g in clean_gaps:
    count, ids = pubmed_search(g["novelty_query"])
    titles = fetch_titles(ids[:10])
    if count == 0:
        verdict, comment, hj = "A未见报道", "全库0命中", []
    else:
        resp = client.chat.completions.create(
            model=MODEL, temperature=0,
            messages=[
                {"role": "system", "content": JUDGE_PROMPT},
                {"role": "user", "content":
                 f"Gap主张：{g['description']}\n\n命中文献：\n" +
                 "\n".join(f"{pid}: {t}" for pid, t in titles.items())}
            ],
        )
        j = json.loads(re.search(r'\{.*\}', resp.choices[0].message.content, re.DOTALL).group())
        verdict, comment, hj = j.get("overall","?"), j.get("comment",""), j.get("hits",[])
    novelty_log.append({"gap_id": g["gap_id"], "description": g["description"],
                        "query": g["novelty_query"], "hit_count": count,
                        "hits": hj, "verdict": verdict, "comment": comment})
    print(f"[{g['gap_id']}] 命中{count:>3}篇 → {verdict} | {g['description'][:36]}")
    time.sleep(1)

with open(f"{SAVE_DIR}/batch02_gaps.json", "w", encoding="utf-8") as f:
    json.dump(clean_gaps, f, ensure_ascii=False, indent=2)
with open(f"{SAVE_DIR}/batch02_novelty_log.json", "w", encoding="utf-8") as f:
    json.dump(novelty_log, f, ensure_ascii=False, indent=2)
print("\n已存 batch02_gaps.json / batch02_novelty_log.json")

Gap原料: 56篇原始研究

生成12条，通过12条
[1] 命中  8篇 → C基本已有人做 | ROS-responsive delivery of exosomes 
[2] 命中  0篇 → A未见报道 | No reported smart敷料 combines biofilm
[3] 命中  2篇 → C基本已有人做 | Despite extensive use of copper (Cu²
[4] 命中 41篇 → C基本已有人做 | All published thermosensitive hydrog
[5] 命中  0篇 → A未见报道 | No intelligent敷料 reported to date ta
[6] 命中  0篇 → A未见报道 | Although light-responsive oxygen-gen
[7] 命中  0篇 → A未见报道 | No dual-enzyme-responsive (MMP-9 + N
[8] 命中  0篇 → A未见报道 | Despite strong rationale for combini
[9] 命中  2篇 → C基本已有人做 | All current exosome-loaded hydrogels
[10] 命中 11篇 → A未见报道 | No smart敷料 reported uses *hypoxia-re
[11] 命中 11篇 → C基本已有人做 | While multiple hydrogels load growth
[12] 命中3433篇 → B沾边但未完全覆盖 | No intelligent敷料 integrates *real-ti

已存 batch02_gaps.json / batch02_novelty_log.json


In [35]:
for g in clean_gaps:
    n = next(x for x in novelty_log if x["gap_id"] == g["gap_id"])
    if not n["verdict"].startswith("A"):
        continue
    print("="*70)
    print(f"[{g['gap_id']}] {g['type']} | 可行性:{g['feasibility']} | 查重命中:{n['hit_count']}篇")
    print(f"描述: {g['description']}")
    print(f"理由: {g['rationale']}")
    print(f"支撑文献: {g['supporting_pmids']}")
    print(f"查重检索式: {g['novelty_query']}\n")

[2] 负载空白 | 可行性:中 | 查重命中:0篇
描述: No reported smart敷料 combines biofilm-disrupting enzymatic activity (e.g., DNase I, dispersin B, alginate lyase) with real-time, enzyme-responsive (MMP/NE/elastase) release — while MMP/enzyme-responsive systems exist (PMID35857459, PMID37420287), they deliver antimicrobials or growth factors, not enzymes targeting biofilm matrix components.
理由: Biofilms in chronic wounds are structurally reinforced by eDNA, polysaccharides, and proteins; yet current enzyme-responsive hydrogels release effectors that act *on cells*, not *on the biofilm scaffold*. Co-loading and enzyme-triggered release of biofilm-degrading enzymes would enable autonomous, microenvironment-activated biofilm clearance — a critical unmet need for MRSA/P. aeruginosa-infected chronic wounds.
支撑文献: ['35857459', '37420287']
查重检索式: ("biofilm enzyme" OR "DNase I" OR "dispersin B" OR "alginate lyase") AND (hydrogel) AND ("MMP-responsive" OR "elastase-responsive" OR "neutrophil elastase-responsive") A

In [36]:
SELECTED_GAPS = [2, 7, 8]

def build_evidence_bundle(gap, src_results, src_papers):
    bundle = {"gap": gap["description"], "novelty": "全库查重未见报道", "papers": []}
    for pmid in gap["supporting_pmids"]:
        ext = next((r_ for r_ in src_results if str(r_["pmid"]) == str(pmid)), None)
        pap = next((p for p in src_papers if str(p["pmid"]) == str(pmid)), None)
        if ext and pap:
            bundle["papers"].append({
                "pmid": str(pmid), "title": pap["title"],
                "abstract": pap["abstract"][:1500],
                "extracted": {k: v for k, v in ext.items()
                              if k not in ("pmid","title","doi","year","evidence")},
                "evidence_quotes": ext.get("evidence", {}),
            })
    return bundle

hypotheses = []
for g in clean_gaps:
    if g["gap_id"] not in SELECTED_GAPS:
        continue
    bundle = build_evidence_bundle(g, results_all, papers_all)
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.4,
        messages=[{"role": "system", "content": HYPO_PROMPT},
                  {"role": "user", "content": json.dumps(bundle, ensure_ascii=False)}],
    )
    h = json.loads(re.search(r'\{.*\}', resp.choices[0].message.content, re.DOTALL).group())
    h["gap_id"] = g["gap_id"]
    hypotheses.append(h)
    print(f"✓ Gap[{g['gap_id']}] 假设生成，证据链{len(h.get('evidence_chain',[]))}环")
    time.sleep(1)

✓ Gap[2] 假设生成，证据链4环
✓ Gap[7] 假设生成，证据链4环
✓ Gap[8] 假设生成，证据链4环


In [37]:
# 重建原文索引（全量77篇）
corpus_text = {}
for p in papers_all:
    full = p["title"] + " " + p["abstract"] + " " + " ".join(s_["text"] for s_ in p["sections"])
    corpus_text[str(p["pmid"])] = normalize(full)

for h in hypotheses:
    ok, bad = 0, []
    for link in h.get("evidence_chain", []):
        q = normalize(str(link.get("quote", "")))
        src = corpus_text.get(str(link.get("pmid")), "")
        if q in src or (q and difflib_ok(q, src)):
            ok += 1
        else:
            bad.append(link)
    h["evidence_chain"] = [l for l in h.get("evidence_chain", []) if l not in bad]
    print(f"Gap[{h['gap_id']}] {ok}环通过，{len(bad)}环剔除")

with open(f"{SAVE_DIR}/batch02_hypotheses.json", "w", encoding="utf-8") as f:
    json.dump(hypotheses, f, ensure_ascii=False, indent=2)

Gap[2] 4环通过，0环剔除
Gap[7] 4环通过，0环剔除
Gap[8] 4环通过，0环剔除


In [38]:
def fmt_items(x, prefix="  - "):
    """把list/dict格式的字段排成人类可读列表"""
    if isinstance(x, list):
        return "\n".join(prefix + str(i) for i in x)
    if isinstance(x, dict):
        return "\n".join(f"  · **{k}**：\n" + "\n".join("    - " + str(i) for i in v)
                         if isinstance(v, list) else f"  · **{k}**：{v}"
                         for k, v in x.items())
    return str(x)

from datetime import date
L = []
L.append("# 慢性难愈创面智能敷料的文献驱动科学发现：调研报告\n")
L.append(f"> 生成日期：{date.today()}｜语料库：PubMed 77篇开放全文（2019–2026，含定向补充检索）")
L.append("> 本报告由文献发现Agent自动生成并经人工审核；仅用于科学研究，不构成任何临床诊疗建议。\n")

L.append("\n## 1. 科学问题\n")
L.append("慢性难愈创面（糖尿病足溃疡、静脉性溃疡、压疮）的愈合停滞与微环境异常"
         "（高ROS、蛋白酶失衡、生物膜感染、巨噬细胞极化障碍）密切相关。智能响应型敷料"
         "能感知微环境并按需释药，但材料体系、响应机制与生物活性负载之间的组合空间"
         "远未被系统探索。本研究构建文献驱动的发现型Agent，从公开文献中自动识别研究空白"
         "并生成可验证假设。\n")

L.append("\n## 2. 方法：发现管线\n")
L.append("检索（PubMed E-utilities）→ 全文结构化解析（PMC XML）→ LLM实体抽取"
         "（9字段schema，逐字段附原文证据句）→ Gap系统扫描（PMID白名单校验）→ "
         "全库新颖性查重（检索式留档）→ 假设生成 → 证据链逐字核验。"
         "LLM仅参与内容生成；PMID挂载、引用校验、查重计数均由确定性程序执行。"
         "综述类文献仅作背景，不进入Gap生成原料。\n")

# 第3节：交叉引用矩阵（用77篇全量）
L.append("\n## 3. 文献交叉引用矩阵（材料 × 响应机制）\n")
mat = {}
for r_ in results_all:
    if r_.get("evidence_level") == "review":
        continue
    mat_key = (str(r_.get("material_matrix"))[:50], str(r_.get("response_mechanism")))
    mat.setdefault(mat_key, []).append(str(r_["pmid"]))
L.append("| 材料基体 | 响应机制 | 文献 |")
L.append("|---------|---------|------|")
for (m, mech), pms in sorted(mat.items()):
    L.append(f"| {m} | {mech} | {', '.join(pms)} |")

L.append("\n## 4. Gap清单与新颖性查重\n")
L.append("| # | 类型 | Gap | 查重命中 | 判定 |")
L.append("|---|------|-----|---------|------|")
for g, n in zip(clean_gaps, novelty_log):
    desc = str(g['description']).replace("|", "｜")
    L.append(f"| {g['gap_id']} | {g['type']} | {desc[:60]}… "
             f"| {n['hit_count']}篇 | {n['verdict']} |")
L.append("\n查重检索式与逐条判读记录见 batch02_novelty_log.json（可复核）。\n")

L.append("\n## 5. 有效发现（3条可验证假设）\n")
for h in hypotheses:
    ed = h["experimental_design"]
    L.append(f"\n### 发现 {h['gap_id']}\n")
    L.append(f"**假设**：{h['hypothesis']}\n")
    L.append(f"**机制推理**：{h['mechanism_rationale']}\n")
    L.append(f"**已知 vs 新增**：\n- 已知：{h['known_vs_new']['already_known']}"
             f"\n- 本研究新增：{h['known_vs_new']['our_increment']}\n")
    L.append("**证据链**（逐字核验通过）：")
    for link in h["evidence_chain"]:
        L.append(f"- {link['claim']}（PMID:{link['pmid']}：\"{link['quote'][:100]}…\"）")
    L.append(f"\n**验证方案**：\n- 模型：{ed['models']}\n- 对照组：\n"
             + fmt_items(ed['controls'], "  - ")
             + f"\n- 观察指标：\n" + fmt_items(ed['endpoints'])
             + "\n> 注：上述量化指标为假设性预期目标，非既有结论。\n")

L.append("\n## 6. 附录：数据与日志清单\n")
L.append("- batch01_metadata.csv / batch02_metadata.csv：检索元数据\n"
         "- batch01_fulltext.jsonl / batch02_fulltext.jsonl：结构化全文（77篇）\n"
         "- batch01_extracted.jsonl / batch02_extracted.jsonl：实体抽取（含证据句）\n"
         "- batch02_gaps.json：12条Gap（PMID白名单校验通过）\n"
         "- batch02_novelty_log.json：全库查重日志（检索式可复核）\n"
         "- batch02_hypotheses.json：3条假设（证据链已逐字核验净化）\n")

report = "\n".join(L)
with open(f"{SAVE_DIR}/调研报告_v3.md", "w", encoding="utf-8") as f:
    f.write(report)
print(f"报告已生成：调研报告_v3.md（{len(report)}字符）")

报告已生成：调研报告_v3.md（12966字符）


In [43]:
def fmt_items(x, prefix="  - "):
    """把list/dict格式的字段排成人类可读列表"""
    if isinstance(x, list):
        return "\n".join(prefix + str(i) for i in x)
    if isinstance(x, dict):
        return "\n".join(f"  · **{k}**：\n" + "\n".join("    - " + str(i) for i in v)
                         if isinstance(v, list) else f"  · **{k}**：{v}"
                         for k, v in x.items())
    return str(x)

from datetime import date
L = []
L.append("# 慢性难愈创面智能敷料的文献驱动科学发现：调研报告\n")
L.append(f"> 生成日期：{date.today()}｜语料库：PubMed 77篇开放全文（2019–2026，含定向补充检索）")
L.append("> 本报告由文献发现Agent自动生成并经人工审核；仅用于科学研究，不构成任何临床诊疗建议。\n")

L.append("\n## 1. 科学问题\n")
L.append("慢性难愈创面（糖尿病足溃疡、静脉性溃疡、压疮）的愈合停滞与微环境异常"
         "（高ROS、蛋白酶失衡、生物膜感染、巨噬细胞极化障碍）密切相关。智能响应型敷料"
         "能感知微环境并按需释药，但材料体系、响应机制与生物活性负载之间的组合空间"
         "远未被系统探索。本研究构建文献驱动的发现型Agent，从公开文献中自动识别研究空白"
         "并生成可验证假设。\n")

L.append("\n## 2. 方法：发现管线\n")
L.append("检索（PubMed E-utilities）→ 全文结构化解析（PMC XML）→ LLM实体抽取"
         "（9字段schema，逐字段附原文证据句）→ Gap系统扫描（PMID白名单校验）→ "
         "全库新颖性查重（检索式留档）→ 假设生成 → 证据链逐字核验。"
         "LLM仅参与内容生成；PMID挂载、引用校验、查重计数均由确定性程序执行。"
         "综述类文献仅作背景，不进入Gap生成原料。\n")

# 第3节：交叉引用矩阵（用77篇全量）
L.append("\n## 3. 文献交叉引用矩阵（材料 × 响应机制）\n")
mat = {}
for r_ in results_all:
    if r_.get("evidence_level") == "review":
        continue
    mat_key = (str(r_.get("material_matrix"))[:50], str(r_.get("response_mechanism")))
    mat.setdefault(mat_key, []).append(str(r_["pmid"]))
L.append("| 材料基体 | 响应机制 | 文献 |")
L.append("|---------|---------|------|")
for (m, mech), pms in sorted(mat.items()):
    L.append(f"| {m} | {mech} | {', '.join(pms)} |")

L.append("\n## 4. Gap清单与新颖性查重\n")
L.append("| # | 类型 | Gap | 查重命中 | 判定 |")
L.append("|---|------|-----|---------|------|")
for g, n in zip(clean_gaps, novelty_log):
    desc = str(g['description']).replace("|", "｜")
    L.append(f"| {g['gap_id']} | {g['type']} | {desc[:60]}… "
             f"| {n['hit_count']}篇 | {n['verdict']} |")
L.append("\n查重检索式与逐条判读记录见 batch02_novelty_log.json（可复核）。\n")

L.append("\n## 5. 有效发现（3条可验证假设）\n")
for h in hypotheses:
    ed = h["experimental_design"]
    L.append(f"\n### 发现 {h['gap_id']}\n")
    L.append(f"**假设**：{h['hypothesis']}\n")
    L.append(f"**机制推理**：{h['mechanism_rationale']}\n")
    L.append(f"**已知 vs 新增**：\n- 已知：{h['known_vs_new']['already_known']}"
             f"\n- 本研究新增：{h['known_vs_new']['our_increment']}\n")
    L.append("**证据链**（逐字核验通过）：")
    for link in h["evidence_chain"]:
        L.append(f"- {link['claim']}（PMID:{link['pmid']}：\"{link['quote'][:100]}…\"）")
    L.append(f"\n**验证方案**：\n- 模型：{ed['models']}\n- 对照组：\n"
             + fmt_items(ed['controls'], "  - ")
             + f"\n- 观察指标：\n" + fmt_items(ed['endpoints'])
             + "\n> 注：上述量化指标为假设性预期目标，非既有结论。\n")

L.append("\n## 6. 附录：数据与日志清单\n")
L.append("- batch01_metadata.csv / batch02_metadata.csv：检索元数据\n"
         "- batch01_fulltext.jsonl / batch02_fulltext.jsonl：结构化全文（77篇）\n"
         "- batch01_extracted.jsonl / batch02_extracted.jsonl：实体抽取（含证据句）\n"
         "- batch02_gaps.json：12条Gap（PMID白名单校验通过）\n"
         "- batch02_novelty_log.json：全库查重日志（检索式可复核）\n"
         "- batch02_hypotheses.json：3条假设（证据链已逐字核验净化）\n")

report = "\n".join(L)
with open(f"{SAVE_DIR}/调研报告_v5.md", "w", encoding="utf-8") as f:
    f.write(report)
print(f"报告已生成：调研报告_v4.md（{len(report)}字符）")

报告已生成：调研报告_v4.md（12968字符）


In [44]:
h2 = next(h for h in hypotheses if h["gap_id"] == 2)
h2["experimental_design"]["endpoints"] = (
    "主要：第7天创面细菌载量（CFU/g）、生物膜厚度（CLSM定量eDNA/PNAG荧光强度）、"
    "再上皮化率（H&E面积占比）；次要：MMP-9活性（FRET探针法）、"
    "创面组织中DNase I/dispersin B活性保留率（比色法）、"
    "CD31⁺血管密度、α-SMA⁺肌成纤维细胞数量"
)

# 顺手修发现8第3环：把引句换成该文中真正谈血管生成的句子
h8 = next(h for h in hypotheses if h["gap_id"] == 8)
candidate = "promoting angiogenesis through NO-mediated endothelial cell activation"
if difflib_ok(normalize(candidate), corpus_text.get("40810663", "")):
    h8["evidence_chain"][2] = {
        "claim": "NO介导的内皮细胞激活可促进血管生成",
        "pmid": "40810663", "quote": candidate}
    print("✓ 发现8第3环已替换并通过核验")
else:
    print("⚠ 新引句在原文未找到，保持原样（原引句也是真实的，只是论点偏移）")

with open(f"{SAVE_DIR}/batch02_hypotheses.json", "w", encoding="utf-8") as f:
    json.dump(hypotheses, f, ensure_ascii=False, indent=2)

✓ 发现8第3环已替换并通过核验


In [39]:
TRANSLATE_PROMPT = """把以下研究Gap的描述翻译成流畅的中文。
规则：PMID编号保留原样；专业术语保留英文原名并括注中文（如 dispersin B（分散素B））；
markdown星号去掉；只输出JSON数组，与输入一一对应：
[{"gap_id": 1, "description_cn": "...", "rationale_cn": "..."}]"""

resp = client.chat.completions.create(
    model=MODEL, temperature=0,
    messages=[{"role": "system", "content": TRANSLATE_PROMPT},
              {"role": "user", "content": json.dumps(
                  [{"gap_id": g["gap_id"], "description": g["description"],
                    "rationale": g.get("rationale","")} for g in clean_gaps],
                  ensure_ascii=False)}],
)
trans = {t["gap_id"]: t for t in json.loads(
    re.search(r'\[.*\]', resp.choices[0].message.content, re.DOTALL).group())}
for g in clean_gaps:
    if g["gap_id"] in trans:
        g["description"] = trans[g["gap_id"]]["description_cn"]
        g["rationale"] = trans[g["gap_id"]].get("rationale_cn", g.get("rationale",""))
print("翻译完成，示例：", clean_gaps[1]["description"][:60])

翻译完成，示例： 目前尚无报道的智能敷料将生物膜破坏性酶活性（如DNase I、分散素B（dispersin B）、海藻酸盐裂解酶（alg


In [40]:
h7 = next(h for h in hypotheses if h["gap_id"] == 7)
for key in ["hypothesis", "mechanism_rationale"]:
    h7[key] = h7[key].replace("VAA", "AAPV").replace("PLGLAG-VAA", "PLGLAG-AAPV")
h7["experimental_design"]["controls"] = [
    c.replace("VAA", "AAPV") for c in h7["experimental_design"]["controls"]]
# 已知vs新增里也同步
for k in h7["known_vs_new"]:
    h7["known_vs_new"][k] = h7["known_vs_new"][k].replace("VAA", "AAPV")

h2 = next(h for h in hypotheses if h["gap_id"] == 2)
h2["experimental_design"]["endpoints"] = str(
    h2["experimental_design"]["endpoints"]).replace("dispersionsin", "dispersin")

with open(f"{SAVE_DIR}/batch02_hypotheses.json", "w", encoding="utf-8") as f:
    json.dump(hypotheses, f, ensure_ascii=False, indent=2)
print("已修正并存盘")

已修正并存盘
